# Laboratori 3 — Analiza e Kompleksitetit Kohor Mbi Algoritmet e Renditjes

Në këtë laborator do të analizojmë në mënyrë vizuale tre algoritmet e renditjes:
**Insertion Sort**, **Selection Sort** dhe **Merge Sort**.

Ushtrimi 1 — Vizualizim Pygame: Numri i Krahasimeve \
Ushtrimi 2 — Krahasim Empirik: 3 Raste × 3 Algoritme (Plotly) \
Ushtrimi 3 — Kurba e Rritjes: O(n²) vs O(n log n) (Plotly) \
Ushtrimi 4 — Lojë: Gjej Algoritmin!

---

## Ushtrimi 1 — Vizualizim: Rendi i Rritjes së Krahasimeve

### Çfarë është Kompleksiteti Kohor?

> *"Kompleksiteti kohor shprehet si funksion i madhësisë së inputit. Fokusi vendoset si rritet kompleksiteti kur inputi rritet pa kufi."*  

**Kompleksiteti kohor ≠ numri total i operacioneve.**  
**Kompleksiteti kohor = rendi i rritjes së numrit të operacioneve në varësi të n.**

Big O injoron konstantet dhe termat e vegjël — fokusohet vetëm te sjellja kur n → ∞:

| Notacion | Rendi i rritjes | n=10 | n=100 | n=1,000 | n=10,000 |
|---|---|---|---|---|---|
| O(1) | Konstant | 1 | 1 | 1 | 1 |
| O(n) | Linear | 10 | 100 | 1,000 | 10,000 |
| O(n log n) | Linearitmik | ~33 | ~664 | ~9,966 | ~132,877 |
| O(n²) | Kuadratik | 100 | 10,000 | 1,000,000 | 100,000,000 |

### Çfarë mat vizualizimi?

Ky vizualizim mat **krahasimet** (`a[j] > k?`) — jo shkëmbimet, jo kohën në sekonda.  
Krahasimet zgjidhen si metrikë kryesore sepse:
1. Janë **operacioni themelor i vendimmarrjes** i çdo algoritmi renditjeje
2. Janë baza e **kufirit teorik** për renditjet bazuar në krahasime
3. Janë të **pavarura nga hardware-i** — njësoj në çdo kompjuter

In [8]:
import pygame
import random
import os
import math

# Input
raw = input("Numri i elementeve (Enter = 60): ").strip()
try:
    n = max(2, min(500, int(raw)))
except ValueError:
    n = 60

lista_origjinale = random.sample(range(1, n * 3 + 1), n)

# Algoritmet
# Secili algoritëm ruan hapat: (krahasime, lista, indeksi_a, indeksi_b, i_renditur_deri)

def hapat_insertion(src):
    lst   = src[:]
    hapat = [(0, lst[:], -1, -1, 0)]
    c     = 0
    for i in range(1, len(lst)):
        key = lst[i]
        j   = i - 1
        while j >= 0 and lst[j] > key:
            c += 1
            lst[j + 1] = lst[j]
            hapat.append((c, lst[:], j + 1, j, i))
            j -= 1
        if j >= 0:
            c += 1
        lst[j + 1] = key
        hapat.append((c, lst[:], j + 1, -1, i))
    hapat.append((c, lst[:], -1, -1, len(lst) - 1))
    return hapat

def hapat_selection(src):
    lst   = src[:]
    hapat = [(0, lst[:], -1, -1, 0)]
    c     = 0
    for i in range(len(lst) - 1):
        min_idx = i
        for j in range(i + 1, len(lst)):
            c += 1
            if lst[j] < lst[min_idx]:
                min_idx = j
            hapat.append((c, lst[:], i, j, i - 1))
        if min_idx != i:
            lst[i], lst[min_idx] = lst[min_idx], lst[i]
        hapat.append((c, lst[:], i, min_idx, i))
    hapat.append((c, lst[:], -1, -1, len(lst) - 1))
    return hapat

def hapat_merge(src):
    lst   = src[:]
    hapat = [(0, lst[:], -1, -1, 0)]
    c     = 0

    def bashko(lo, mid, hi):
        nonlocal c
        majta  = lst[lo:mid]
        djathta = lst[mid:hi]
        i = j = 0
        k = lo
        while i < len(majta) and j < len(djathta):
            c += 1
            if majta[i] <= djathta[j]:
                lst[k] = majta[i]
                i += 1
            else:
                lst[k] = djathta[j]
                j += 1
            hapat.append((c, lst[:], k, -1, hi - 1))
            k += 1
        lst[k:hi] = majta[i:] + djathta[j:]
        hapat.append((c, lst[:], -1, -1, hi - 1))

    madhesia = 1
    while madhesia < len(lst):
        for lo in range(0, len(lst), 2 * madhesia):
            mid = min(lo + madhesia,     len(lst))
            hi  = min(lo + 2 * madhesia, len(lst))
            if mid < hi:
                bashko(lo, mid, hi)
        madhesia *= 2

    hapat.append((c, lst[:], -1, -1, len(lst) - 1))
    return hapat

# Llogarit të dhënat
te_gjitha_hapat = [
    hapat_insertion(lista_origjinale),
    hapat_selection(lista_origjinale),
    hapat_merge(lista_origjinale),
]

krahasimet_totale = [h[-1][0] for h in te_gjitha_hapat]
hapat_max         = max(len(h) for h in te_gjitha_hapat)
madhesitë         = list(range(2, n + 1))

# Kurba e kompleksitetit: krahasimet reale për çdo nënmadheësi 2..n
kurba = [
    [hapat_insertion(lista_origjinale[:z])[-1][0] for z in madhesitë],
    [hapat_selection(lista_origjinale[:z])[-1][0] for z in madhesitë],
    [hapat_merge    (lista_origjinale[:z])[-1][0] for z in madhesitë],
]
max_y = max(max(k) for k in kurba)

# Vlerat teorike
teorike = [
    [z * (z - 1) // 2          for z in madhesitë],   # n(n-1)/2  — rastin me të keq
    [z * (z - 1) // 2          for z in madhesitë],   # gjithmonë n(n-1)/2
    [int(z * math.log2(z))     for z in madhesitë],   # n·log₂n
]

print(f"Insertion={krahasimet_totale[0]}  "      f"Selection={krahasimet_totale[1]}  "      f"Merge={krahasimet_totale[2]}")

# Pygame
pygame.init()
screen = pygame.display.set_mode((1500, 860), pygame.RESIZABLE)
pygame.display.set_caption(f"Ushtrimi 1 — Kompleksiteti Kohor  (n={n})")

# Ngjyrat
SFOND        = (14, 14, 26)
PANEL        = (22, 22, 40)
GRAFIK_SFOND = (18, 18, 35)
I_PARENDITUR = (55, 55, 95)
I_RENDITUR   = (72, 199, 142)
AKTIV_A      = (255, 90, 80)
AKTIV_B      = (255, 210, 60)
TEKST        = (220, 220, 240)
ZBEHUR       = (110, 110, 140)

NGJYRAT_ALGO = [(255, 140, 70), (80, 185, 255), (80, 225, 120)]
EMRAT_ALGO   = [
    ("Insertion Sort", "O(n^2)"),
    ("Selection Sort", "O(n^2)"),
    ("Merge Sort",     "O(n log n)"),
]

font_i_madh  = pygame.font.Font(None, 34)
font_mesatar = pygame.font.Font(None, 27)
font_i_vogel = pygame.font.Font(None, 21)
font_mono    = pygame.font.Font(None, 20)

# Gjendja e animacionit
frame     = 0
running   = True
auto_play = False
shpejtesi = 1

# Funksionet e grafikut
def vizato_grafiket(siperfaqe, rect, lst, a, b, i_renditur_deri):
    rx, ry, gjeresi, lartesi = rect
    pygame.draw.rect(siperfaqe, PANEL, rect, border_radius=8)

    gjeresia_shiritit = max(1, gjeresi // len(lst))
    hapesira          = max(0, gjeresia_shiritit // 10)
    vlera_min         = min(lst)
    vlera_max         = max(lst)
    shkalla           = (lartesi - 8) / (vlera_max - vlera_min) if vlera_max > vlera_min else lartesi - 8

    for i, vlera in enumerate(lst):
        x           = rx + i * gjeresia_shiritit + hapesira
        lartesia_sh = int((vlera - vlera_min) * shkalla) + 3
        gj          = max(1, gjeresia_shiritit - hapesira * 2)
        if i == a:
            ngjyra = AKTIV_A
        elif i == b:
            ngjyra = AKTIV_B
        elif i <= i_renditur_deri:
            ngjyra = I_RENDITUR
        else:
            ngjyra = I_PARENDITUR
        pygame.draw.rect(siperfaqe, ngjyra,
                         (x, ry + lartesi - lartesia_sh, gj, lartesia_sh),
                         border_radius=min(2, gj // 3))
        if gjeresia_shiritit >= 20:
            t = font_mono.render(str(vlera), True, TEKST)
            siperfaqe.blit(t, (x + gj // 2 - t.get_width() // 2,
                                ry + lartesi - lartesia_sh - 13))

def vizato_grafik_kompleksiteti(siperfaqe, rect, algo_idx, frame):
    rx, ry, gjeresi, lartesi = rect
    pygame.draw.rect(siperfaqe, GRAFIK_SFOND, rect, border_radius=6)
    pygame.draw.rect(siperfaqe, ZBEHUR,       rect, 1, border_radius=6)

    if max_y == 0 or len(madhesitë) < 2:
        return
    padding_majtas = 38
    padding_posht  = 18
    gx = rx + padding_majtas
    gy = ry + 4
    gw = gjeresi - padding_majtas - 4
    gh = lartesi  - padding_posht  - 4

    def ne_pixel(i, c):
        x = gx + int(i / (len(madhesitë) - 1) * gw)
        y = gy + gh - int(c / max_y * gh)
        return (x, y)
    # Rrjeti horizontal
    for fraksion in [0.25, 0.50, 0.75, 1.00]:
        yy = gy + gh - int(fraksion * gh)
        pygame.draw.line(siperfaqe, (35, 35, 60), (gx, yy), (gx + gw, yy), 1)
        etiketë = font_mono.render(f"{int(fraksion * max_y):,}", True, (70, 70, 100))
        siperfaqe.blit(etiketë, (rx + 1, yy - 6))

    # Boshti X — etiketat e madhësisë n
    for fraksion in [0, 0.25, 0.50, 0.75, 1.00]:
        xi      = int(fraksion * (len(madhesitë) - 1))
        sx, _   = ne_pixel(xi, 0)
        etiketë = font_mono.render(str(madhesitë[xi]), True, (70, 70, 100))
        siperfaqe.blit(etiketë, (sx - 10, gy + gh + 3))

    # Kurba teorike (gri)
    pikat_teorike = [ne_pixel(i, teorike[algo_idx][i]) for i in range(len(madhesitë))]
    if len(pikat_teorike) >= 2:
        pygame.draw.lines(siperfaqe, (55, 55, 75), False, pikat_teorike, 1)

    # Kurba empirike
    pikat_empirike = [ne_pixel(i, kurba[algo_idx][i]) for i in range(len(madhesitë))]
    if len(pikat_empirike) >= 2:
        pygame.draw.lines(siperfaqe, NGJYRAT_ALGO[algo_idx], False, pikat_empirike, 2)

    # Pika aktuale
    ci = min(frame, len(madhesitë) - 1)
    pygame.draw.circle(siperfaqe, NGJYRAT_ALGO[algo_idx],
                       ne_pixel(ci, kurba[algo_idx][ci]), 5)

    # Etiketa
    etiketë = font_mono.render(
        f"krahasime n={n}: {krahasimet_totale[algo_idx]}  (gri=teorike)",
        True, NGJYRAT_ALGO[algo_idx])
    siperfaqe.blit(etiketë, (gx + 4, ry + 4))

def vizato():
    W, H = screen.get_size()
    screen.fill(SFOND)

    # Titulli
    titulli = font_i_madh.render(
        f"Ushtrimi 1 — Kompleksiteti Kohor: Krahasime  (n={n})", True, TEKST)
    screen.blit(titulli, (W // 2 - titulli.get_width() // 2, 8))

    # Mesazhi i gjendjes
    mbaroi = frame >= hapat_max - 1
    if mbaroi:
        msg = (f"Mbaroi!  Ins={krahasimet_totale[0]}  "               f"Sel={krahasimet_totale[1]}  "               f"Merge={krahasimet_totale[2]}  |  SPACE: rivendos  Q: del")
        ngjyra_msg = I_RENDITUR
    elif auto_play:
        perqindja  = frame * 100 // (hapat_max - 1)
        msg        = f"Ekzekutim... {perqindja}%  |  SPACE: ndalon  +/-: shpejtësia ({shpejtesi})  Q: del"
        ngjyra_msg = TEKST
    else:
        msg        = f"SPACE: nis/ndalon   +/-: shpejtësia ({shpejtesi} hap/tick)   Q: del"
        ngjyra_msg = ZBEHUR

    h = font_i_vogel.render(msg, True, ngjyra_msg)
    screen.blit(h, (W // 2 - h.get_width() // 2, 44))

    # Tre panelet
    marzhi        = 10
    gjeresia_pan  = (W - 40) // 3
    fillimi_lart  = 68
    lartesia_bar  = int((H - fillimi_lart - marzhi) * 0.55)
    lartesia_graf = int((H - fillimi_lart - marzhi) * 0.35)

    for p in range(3):
        px = marzhi + p * (gjeresia_pan + marzhi)
        ngjyra_algo = NGJYRAT_ALGO[p]

        # Sfond dhe vija e ngjyrës
        pygame.draw.rect(screen, PANEL,
                         (px, fillimi_lart, gjeresia_pan, lartesia_bar + lartesia_graf + 60),
                         border_radius=10)
        pygame.draw.rect(screen, ngjyra_algo,
                         (px, fillimi_lart, gjeresia_pan, 3),
                         border_radius=2)

        # Emri i algoritmit
        emri, kompleksiteti = EMRAT_ALGO[p]
        t_emri = font_mesatar.render(emri + "  ",    True, ngjyra_algo)
        t_komp = font_mesatar.render(kompleksiteti, True, ZBEHUR)
        tx = px + gjeresia_pan // 2 - (t_emri.get_width() + t_komp.get_width()) // 2
        screen.blit(t_emri, (tx,                       fillimi_lart + 7))
        screen.blit(t_komp, (tx + t_emri.get_width(),  fillimi_lart + 7))

        # Numri i krahasimeve
        hapi = te_gjitha_hapat[p][min(frame, len(te_gjitha_hapat[p]) - 1)]
        krahasimet, lst, a, b, i_renditur = hapi

        t_cmps = font_i_madh.render(f"{krahasimet:,}", True, ngjyra_algo)
        screen.blit(t_cmps, (px + 10, fillimi_lart + 30))
        screen.blit(font_i_vogel.render("krahasime", True, ZBEHUR),
                    (px + 16 + t_cmps.get_width(), fillimi_lart + 38))

        # Vlera teorike
        teo_txt = [
            f"max = {n * (n - 1) // 2}",
            f"n(n-1)/2 = {n * (n - 1) // 2}",
            f"n·log₂n ≈ {int(n * math.log2(n))}",
        ]
        screen.blit(font_i_vogel.render(f"Teorike: {teo_txt[p]}", True, ZBEHUR),
                    (px + 10, fillimi_lart + 56))

        # Statusi "E RENDITUR"
        if frame >= len(te_gjitha_hapat[p]) - 1:
            d = font_mesatar.render("E RENDITUR", True, I_RENDITUR)
            screen.blit(d, (px + gjeresia_pan - d.get_width() - 10, fillimi_lart + 38))

        # Grafiket
        rect_grafiket = (px + 5, fillimi_lart + 76, gjeresia_pan - 10, lartesia_bar)
        vizato_grafiket(screen, rect_grafiket, lst, a, b, i_renditur)

        # Grafiku i kompleksitetit
        rect_grafik = (px + 5,
                       fillimi_lart + 76 + lartesia_bar + 8,
                       gjeresia_pan - 10,
                       lartesia_graf)
        vizato_grafik_kompleksiteti(screen, rect_grafik, p, frame)
        screen.blit(
            font_i_vogel.render("X=n  Y=krahasime  (gri=teorike)", True, ZBEHUR),
            (px + 8, fillimi_lart + 76 + lartesia_bar + 8 + lartesia_graf - 16))
    pygame.display.flip()

# Loop kryesore
clock = pygame.time.Clock()
vizato()

while running:
    clock.tick(60)
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        elif event.type == pygame.KEYDOWN:
            if event.key in (pygame.K_q, pygame.K_ESCAPE):
                running = False
            elif event.key == pygame.K_SPACE:
                if frame >= hapat_max - 1:
                    frame = 0
                else:
                    auto_play = not auto_play
            elif event.key in (pygame.K_EQUALS, pygame.K_PLUS):
                shpejtesi = min(shpejtesi + 1, 200)
            elif event.key == pygame.K_MINUS:
                shpejtesi = max(shpejtesi - 1, 1)
    if auto_play and frame < hapat_max - 1:
        frame = min(frame + shpejtesi, hapat_max - 1)
        if frame >= hapat_max - 1:
            auto_play = False

    vizato()
pygame.quit()
os._exit(0)

pygame 2.6.1 (SDL 2.28.4, Python 3.13.5)
Hello from the pygame community. https://www.pygame.org/contribute.html
Insertion=23309  Selection=44850  Merge=2223


: 

---

## Ushtrimi 2 — Krahasim Empirik: 3 Raste × 3 Algoritme

### Qëllimi
Të vëzhgojmë si ndryshon **numri i krahasimeve** për të tre algoritmet
kur lista është **rastësore**, **e renditur** dhe **e anasjelltë (reverse)**.

### Çfarë tregon grafiku?
- **Selection Sort** — tre vija mbi njëra-tjetrën (pse? — numri i krahasimeve nuk ndryshon kurrë)
- **Insertion Sort** — hendeku i madh midis rasteve: listë e renditur → O(n), reverse → O(n²)
- **Merge Sort** — tre vija pothuajse identike

### Lidhja me teorinë
Vija gri = vlera teorike. Sa afër i qëndron kurba empirike teorisë?

In [1]:
import random, math
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def ins_cmps(src):
    a, c = src[:], 0
    for i in range(1, len(a)):
        k, j = a[i], i-1
        while j >= 0 and a[j] > k: c += 1; a[j+1] = a[j]; j -= 1
        if j >= 0: c += 1
        a[j+1] = k
    return c

def sel_cmps(src):
    a, c = src[:], 0
    for i in range(len(a)-1):
        m = i
        for j in range(i+1, len(a)): c += 1; m = j if a[j] < a[m] else m
        a[i], a[m] = a[m], a[i]
    return c

def merge_cmps(src):
    a, c = src[:], 0
    def mg(lo, mid, hi):
        nonlocal c
        L, R, i, j, k = a[lo:mid], a[mid:hi], 0, 0, lo
        while i < len(L) and j < len(R):
            c += 1
            if L[i] <= R[j]: a[k] = L[i]; i += 1
            else:             a[k] = R[j]; j += 1
            k += 1
        a[k:hi] = L[i:] + R[j:]
    sz = 1
    while sz < len(a):
        for lo in range(0, len(a), 2*sz):
            mid, hi = min(lo+sz, len(a)), min(lo+2*sz, len(a))
            if mid < hi: mg(lo, mid, hi)
        sz *= 2
    return c

madhesite = [10, 50, 100, 200, 500]
raste     = ["Random", "E renditur", "Reverse"]
algoritme = [
    ("Insertion Sort", ins_cmps,   "#FF8C46"),
    ("Selection Sort", sel_cmps,   "#50B9FF"),
    ("Merge Sort",     merge_cmps, "#50E178"),
]

fig = make_subplots(rows=1, cols=3,
    subplot_titles=[nm for nm,_,_ in algoritme], shared_yaxes=False)

for ci, (nm, fn, color) in enumerate(algoritme):
    for rast in raste:
        y_vals = []
        for n in madhesite:
            if rast == "Random":       lst = random.sample(range(1, n*3+1), n)
            elif rast == "E renditur": lst = list(range(1, n+1))
            else:                      lst = list(range(n, 0, -1))
            y_vals.append(fn(lst))
        dashes = {"Random": "solid", "E renditur": "dash", "Reverse": "dot"}
        fig.add_trace(go.Scatter(
            x=madhesite, y=y_vals, name=rast,
            mode="lines+markers",
            line=dict(width=2, dash=dashes[rast]),
            marker=dict(size=7),
            showlegend=(ci == 0),
            hovertemplate=f"{nm}<br>n=%{{x}}<br>krahasime=%{{y}}<extra>{rast}</extra>"
        ), row=1, col=ci+1)
    theo = [n*(n-1)//2 for n in madhesite] if ci < 2 else [int(n*math.log2(n)) for n in madhesite]
    fig.add_trace(go.Scatter(
        x=madhesite, y=theo, name="Teorike",
        mode="lines", line=dict(color="gray", width=1, dash="dot"),
        showlegend=(ci == 0),
        hovertemplate="Teorike: %{y}<extra></extra>"
    ), row=1, col=ci+1)

fig.update_layout(
    title="Ushtrimi 2 — Krahasime sipas rastit: Random / E renditur / Reverse",
    height=500, width=1100, plot_bgcolor="white",
    legend=dict(x=0.5, y=-0.18, orientation="h", xanchor="center"),
    hoverlabel=dict(bgcolor="#1e1e2e", font_color="white")
)
fig.update_xaxes(title_text="n (madhësia)", gridcolor="#eee")
fig.update_yaxes(title_text="Krahasime", gridcolor="#eee")
fig.show()

---

## Ushtrimi 3 — Kurba e Rritjes: O(n²) vs O(n log n)

### Qëllimi
Të vizualizojmë si rritet numri i krahasimeve kur n rritet —
dhe të kuptojmë pse diferenca midis O(n²) dhe O(n log n) bëhet dramatike.

### Çfarë tregon grafiku?
Të gjithë algoritmet mbi të njëjtat boshte — kjo tregon vizualisht:
- O(n²): parabolë — dyfishimi i n → 4x më shumë krahasime
- O(n log n): pothuajse lineare — dyfishimi i n → pak më shumë se 2x krahasime

In [7]:
import random, math
import plotly.graph_objects as go

def ins_cmps(src):
    a, c = src[:], 0
    for i in range(1, len(a)):
        k, j = a[i], i-1
        while j >= 0 and a[j] > k: c += 1; a[j+1] = a[j]; j -= 1
        if j >= 0: c += 1
        a[j+1] = k
    return c

def sel_cmps(src):
    a, c = src[:], 0
    for i in range(len(a)-1):
        m = i
        for j in range(i+1, len(a)): c += 1; m = j if a[j] < a[m] else m
        a[i], a[m] = a[m], a[i]
    return c

def merge_cmps(src):
    a, c = src[:], 0
    def mg(lo, mid, hi):
        nonlocal c
        L, R, i, j, k = a[lo:mid], a[mid:hi], 0, 0, lo
        while i < len(L) and j < len(R):
            c += 1
            if L[i] <= R[j]: a[k] = L[i]; i += 1
            else:             a[k] = R[j]; j += 1
            k += 1
        a[k:hi] = L[i:] + R[j:]
    sz = 1
    while sz < len(a):
        for lo in range(0, len(a), 2*sz):
            mid, hi = min(lo+sz, len(a)), min(lo+2*sz, len(a))
            if mid < hi: mg(lo, mid, hi)
        sz *= 2
    return c

sizes = list(range(10, 501, 10))
print("Duke llogaritur...", end="", flush=True)
ins_y   = [ins_cmps(random.sample(range(1, s*3+1), s)) for s in sizes]
sel_y   = [s*(s-1)//2 for s in sizes]
merge_y = [merge_cmps(random.sample(range(1, s*3+1), s)) for s in sizes]
print(" gati!")

theo_n2    = [s*(s-1)//2          for s in sizes]
theo_nlogn = [int(s*math.log2(s)) for s in sizes]

fig = go.Figure()
fig.add_trace(go.Scatter(x=sizes, y=ins_y,   name="Insertion Sort (random)",      line=dict(color="#FF8C46", width=2)))
fig.add_trace(go.Scatter(x=sizes, y=sel_y,   name="Selection Sort (gjithmone)",   line=dict(color="#50B9FF", width=2)))
fig.add_trace(go.Scatter(x=sizes, y=merge_y, name="Merge Sort (random)",          line=dict(color="#50E178", width=2)))
fig.add_trace(go.Scatter(x=sizes, y=theo_n2,    name="Teorike O(n^2)",    line=dict(color="#aaa", width=1, dash="dot")))
fig.add_trace(go.Scatter(x=sizes, y=theo_nlogn, name="Teorike O(n log n)",line=dict(color="#ccc", width=1, dash="dot")))

fig.update_layout(
    title="Ushtrimi 3 — Kurba e rritjes: O(n^2) vs O(n log n)",
    xaxis=dict(title="n (madhesia e listes)", gridcolor="#eee"),
    yaxis=dict(title="Numri i krahasimeve", gridcolor="#eee"),
    plot_bgcolor="white", height=520, width=1000,
    legend=dict(x=0.01, y=0.99),
    hoverlabel=dict(bgcolor="#1e1e2e", font_color="white")
)
fig.show()

Duke llogaritur...

 gati!


---

## Ushtrimi 4 — Lojë: Gjej Algoritmin!

### Qëllimi
Të aplikojmë njohuritë e kompleksitetit intuitivisht — jo me mbajtje mend, por me logjikë.

### Rregullat
Programi zgjedh rastësisht një algoritëm, e ekzekuton mbi një listë rastësore
dhe tregon vetëm numrin e krahasimeve. Ti duhet të gjendësh cilin algoritëm përdori.

### Strategjia (bazuar në teori)

| Nëse krahasimet janë... | Algoritmi është... | Pse? |
|---|---|---|
| ≈ n(n-1)/2 (referenca e dhënë) | Selection Sort | Gjithmonë saktësisht n(n-1)/2 |
| Shumë më pak se n(n-1)/2 | Insertion Sort | Lista rastësore → ~n²/4 krahasime |
| ≈ n·log₂n (referenca e dhënë) | Merge Sort | Gjithmonë ≈ n·log₂n |

### Çelësat
- I → Insertion Sort &nbsp;|&nbsp; S → Selection Sort &nbsp;|&nbsp; M → Merge Sort
- SPACE → pyetja tjetër (pas përgjigjes) &nbsp;|&nbsp; Q → del

In [ ]:
import pygame, random, math, os

pygame.init()
screen = pygame.display.set_mode((950, 580), pygame.RESIZABLE)
pygame.display.set_caption("Ushtrimi 4 - Gjej Algoritmin!")
fB = pygame.font.Font(None, 52); fM = pygame.font.Font(None, 36)
fS = pygame.font.Font(None, 26); fN = pygame.font.Font(None, 22)
BG = (14,14,26); PB = (22,22,40); TC = (220,220,240); DC = (110,110,140)
OK = (72,199,142); ERR = (255,90,80)
COL   = [(255,140,70), (80,185,255), (80,225,120)]
NAMES = ["Insertion Sort", "Selection Sort", "Merge Sort"]
KEYS  = [pygame.K_i, pygame.K_s, pygame.K_m]

def ins_cmps(src):
    a, c = src[:], 0
    for i in range(1, len(a)):
        k, j = a[i], i-1
        while j >= 0 and a[j] > k: c += 1; a[j+1] = a[j]; j -= 1
        if j >= 0: c += 1
        a[j+1] = k
    return c

def sel_cmps(src):
    a, c = src[:], 0
    for i in range(len(a)-1):
        m = i
        for j in range(i+1, len(a)): c += 1; m = j if a[j] < a[m] else m
        a[i], a[m] = a[m], a[i]
    return c

def merge_cmps(src):
    a, c = src[:], 0
    def mg(lo, mid, hi):
        nonlocal c
        L, R, i, j, k = a[lo:mid], a[mid:hi], 0, 0, lo
        while i < len(L) and j < len(R):
            c += 1
            if L[i] <= R[j]: a[k] = L[i]; i += 1
            else:             a[k] = R[j]; j += 1
            k += 1
        a[k:hi] = L[i:] + R[j:]
    sz = 1
    while sz < len(a):
        for lo in range(0, len(a), 2*sz):
            mid, hi = min(lo+sz, len(a)), min(lo+2*sz, len(a))
            if mid < hi: mg(lo, mid, hi)
        sz *= 2
    return c

FNS = [ins_cmps, sel_cmps, merge_cmps]

def new_q():
    n    = random.choice([15, 20, 25, 30, 40, 50])
    lst  = random.sample(range(1, n*3+1), n)
    algo = random.randint(0, 2)
    return n, lst, algo, FNS[algo](lst), n*(n-1)//2, int(n*math.log2(n))

score, total = 0, 0
n, lst, correct, cmps, tn2, tnl = new_q()
state, answer = "question", -1
clock = pygame.time.Clock()
running = True

def draw():
    W, H = screen.get_size(); screen.fill(BG)
    # Header
    t = fM.render("Ushtrimi 4 — Gjej Algoritmin!", True, TC)
    screen.blit(t, (W//2-t.get_width()//2, 16))
    pct = f"{score*100//total}%" if total > 0 else "—"
    sc = fS.render(f"Rezultati: {score}/{total}  ({pct})", True, OK if total==0 or score/max(total,1)>=0.5 else ERR)
    screen.blit(sc, (W-sc.get_width()-16, 20))

    # Info box
    pygame.draw.rect(screen, PB, (30, 58, W-60, 175), border_radius=10)
    lines = [
        (fM, f"Lista ka   n = {n}   elemente  (rastesore)", TC),
        (fM, f"Algoritmi beri:   {cmps}   krahasime", COL[1]),
        (fS, f"Referenca:   n(n-1)/2 = {tn2}     n*log2(n) ~ {tnl}", DC),
    ]
    for k, (fnt, txt, col) in enumerate(lines):
        s = fnt.render(txt, True, col)
        screen.blit(s, (W//2-s.get_width()//2, 72 + k*50))

    # Mini histogram
    bar_top, bar_h = 242, 100
    bw = max(1, (W-60)//n)
    lo, hi = min(lst), max(lst)
    sc2 = (bar_h-4)/(hi-lo) if hi > lo else bar_h-4
    for i2, v in enumerate(lst):
        bh = int((v-lo)*sc2)+4
        pygame.draw.rect(screen, (55,55,95), (30+i2*bw, bar_top+bar_h-bh, max(1,bw-1), bh), border_radius=1)

    # Answer buttons
    by = 360
    for k, (nm, col) in enumerate(zip(NAMES, COL)):
        bx = 30 + k*((W-60)//3 + 10); bw2 = (W-60)//3
        bc = col
        if state == "feedback":
            if k == correct:              bc = OK
            elif k == answer and answer != correct: bc = ERR
        pygame.draw.rect(screen, PB, (bx, by, bw2, 75), border_radius=10)
        pygame.draw.rect(screen, bc,  (bx, by, bw2, 75), 3, border_radius=10)
        t2 = fM.render(nm, True, col)
        t3 = fN.render(["[ I ]","[ S ]","[ M ]"][k], True, DC)
        screen.blit(t2, (bx+bw2//2-t2.get_width()//2, by+14))
        screen.blit(t3, (bx+bw2//2-t3.get_width()//2, by+48))

    # Feedback / hint
    if state == "feedback":
        if answer == correct:
            msg, col2 = f"Sakte!  Ishte {NAMES[correct]}   —   SPACE per pyetjen tjeter", OK
        else:
            msg, col2 = f"Gabim!  Ishte {NAMES[correct]} ({cmps} krahasime)   —   SPACE per tjeter", ERR
        fb = fS.render(msg, True, col2)
        screen.blit(fb, (W//2-fb.get_width()//2, 460))
    else:
        h = fS.render("I = Insertion     S = Selection     M = Merge", True, DC)
        screen.blit(h, (W//2-h.get_width()//2, 465))
    pygame.display.flip()

draw()
while running:
    clock.tick(60)
    for e in pygame.event.get():
        if e.type == pygame.QUIT: running = False
        elif e.type == pygame.KEYDOWN:
            if e.key in (pygame.K_q, pygame.K_ESCAPE): running = False
            elif state == "question" and e.key in KEYS:
                answer = KEYS.index(e.key); total += 1
                if answer == correct: score += 1
                state = "feedback"; draw()
            elif state == "feedback" and e.key == pygame.K_SPACE:
                n, lst, correct, cmps, tn2, tnl = new_q()
                state = "question"; answer = -1; draw()
    draw()
pygame.quit()
os._exit(0)

pygame 2.6.1 (SDL 2.28.4, Python 3.13.5)
Hello from the pygame community. https://www.pygame.org/contribute.html


: 